# Test du pipeline d'ingestion

Vérifie que le pipeline `src/ingest/` produit **les mêmes chunks** que le notebook d'exploration.

1. Sauvegarde de la sortie du notebook d'exploration (référence)
2. Exécution du pipeline
3. Contrôles rapides sur les chunks
4. Comparaison avec la référence (ids, texte, métadonnées)

In [ ]:
import json
import logging
import shutil

import pandas as pd

from assistant_regles.ingest.chunk import lire_jsonl
from assistant_regles.ingest.config import charger_config
from assistant_regles.ingest.pipeline import executer

logging.basicConfig(level=logging.INFO, format="%(levelname)-7s %(name)s — %(message)s", force=True)

config = charger_config()  # trouve la racine du repo depuis notebooks/
print("PDF   :", config.chemin_pdf, "| existe :", config.chemin_pdf.exists())
print("Cache :", config.chemin_cache_docling, "| existe :", config.chemin_cache_docling.exists())

## 1. Référence : sortie du notebook d'exploration
Le pipeline écrit au même endroit : on copie d'abord l'ancien fichier pour pouvoir comparer.

In [ ]:
reference = config.chemin_chunks.with_name("chunks_notebook.jsonl")
if config.chemin_chunks.exists() and not reference.exists():
    shutil.copy(config.chemin_chunks, reference)
print("Référence disponible :", reference.exists())

## 2. Exécution du pipeline
Recharge le cache Docling s'il existe (sinon conversion complète, plusieurs minutes).

In [ ]:
resultat = executer(config)
resultat

## 3. Contrôles rapides

In [ ]:
chunks = lire_jsonl(resultat.chemin_chunks)
df_chunks = pd.DataFrame([c.model_dump() for c in chunks])

print(df_chunks["nb_tokens"].describe(percentiles=[.5, .9, .95]).round(0))
print("Hors budget          :", df_chunks["hors_budget"].sum())
print("Unités découpées     :", ((df_chunks["partie"] == 1) & (df_chunks["nb_parties"] > 1)).sum())
print("Identifiants uniques :", df_chunks["id"].is_unique)

# Couverture : chaque élément conservé apparaît dans au moins un chunk
df_elements = pd.read_parquet(resultat.chemin_elements)
conserves = set(df_elements.loc[df_elements["motif_exclusion"].isna(), "ordre"])
couverts = {o for c in chunks for o in c.ordres}
print("Éléments non couverts :", len(conserves - couverts))

In [ ]:
print(df_chunks.sample(1, random_state=1)["texte"].iloc[0])

## 4. Comparaison avec la référence
Objectif : 0 id manquant de chaque côté et 0 différence.

In [ ]:
def charger_par_id(chemin):
    with open(chemin, encoding="utf-8") as f:
        return {d["id"]: d for d in map(json.loads, f)}

ref, nouv = charger_par_id(reference), charger_par_id(resultat.chemin_chunks)
print("Chunks notebook / pipeline :", len(ref), "/", len(nouv))
print("Ids seulement dans le notebook :", len(ref.keys() - nouv.keys()))
print("Ids seulement dans le pipeline :", len(nouv.keys() - ref.keys()))

CHAMPS = ["texte", "code", "sous_section", "page_debut", "page_fin", "codes_cites", "nb_tokens"]
differences = [
    {"id": i, "champ": champ, "notebook": ref[i][champ], "pipeline": nouv[i][champ]}
    for i in sorted(ref.keys() & nouv.keys())
    for champ in CHAMPS
    if ref[i][champ] != nouv[i][champ]
]
print("Différences sur les chunks communs :", len(differences))
pd.DataFrame(differences).head(10)